In [8]:
!pip install -q evaluate seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 952.0 kB/s eta 0:00:00a 0:00:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.5 MB/s eta 0:00:00


In [20]:
import torch
from transformers import AutoModelForTokenClassification, AutoTokenizer
import numpy as np
from datasets import load_dataset
import evaluate
from transformers import Trainer
from torch.utils.data import DataLoader
from transformers import DataCollatorForTokenClassification

In [10]:
# 1. Khai báo tên Repo của bạn trên Hugging Face
repo_id = "imbee510/finetuned_ner_xlm_roberta"

# 2. Load model và tokenizer trực tiếp từ Hugging Face
print("Đang tải model và tokenizer từ Hugging Face...")
tokenizer = AutoTokenizer.from_pretrained(repo_id)
model = AutoModelForTokenClassification.from_pretrained(repo_id)

# 3. Đưa model lên GPU (nếu có) để dự đoán nhanh hơn
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval() # Đưa model về chế độ đánh giá (tắt dropout)
id2label = model.config.id2label

Đang tải model và tokenizer từ Hugging Face...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [11]:
text = "I am a student in HCMUS."
inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=128)

# Đưa data lên cùng thiết bị với model (GPU/CPU)
inputs = {k: v.to(device) for k, v in inputs.items()}

In [12]:
inputs

{'input_ids': tensor([[     0,     87,    444,     10,   9836,     23, 111566,   8762,      5,
               2]], device='cuda:0'),
 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], device='cuda:0')}

In [13]:
model.eval()
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits
    # Lấy ID của label có xác suất cao nhất cho từng token
    print(f"Logits shape ban đầu: {outputs.logits.shape}") 
    
    # 2. Cách lấy nhãn "Bất tử" (Dùng .squeeze() để loại bỏ các chiều bằng 1)
    # Nếu là [1, 10, 9] -> nó sẽ về [10, 9]
    # Sau đó argmax dim=-1 -> nó sẽ về [10]
    predictions = torch.argmax(outputs.logits, dim=-1).squeeze().cpu().numpy()

    print(f"Shape sau khi xử lý: {predictions.shape}") # Phải là (10,)

# 6. Hiển thị kết quả (Map Token - Label)
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])


Logits shape ban đầu: torch.Size([1, 10, 9])
Shape sau khi xử lý: (10,)


In [14]:
tokens

['<s>', '▁I', '▁am', '▁a', '▁student', '▁in', '▁HCM', 'US', '.', '</s>']

In [15]:
# Kiểm tra thử:
print(f"Hình dạng của predictions: {predictions.shape}")

Hình dạng của predictions: (10,)


In [16]:
predictions

array([0, 0, 0, 0, 0, 0, 3, 4, 0, 0])

In [17]:
final_results = []

print(f"{'Token':<15} | {'Nhãn dự đoán':<10}")
print("-" * 30)

for i in range(len(tokens)):
    token = tokens[i]
    
    # Bỏ qua các token hệ thống (<s>, </s>)
    if token in tokenizer.all_special_tokens:
        continue
    
    # Lấy nhãn từ ID
    label_id = predictions[i]
    label_name = id2label[label_id]
    
    # Làm sạch ký tự ' ' đặc biệt của XLM-RoBERTa để dễ nhìn
    clean_token = token.replace(' ', ' ')
    
    print(f"{clean_token:<15} | {label_name}")
    final_results.append((clean_token, label_name))

Token           | Nhãn dự đoán
------------------------------
▁I              | O
▁am             | O
▁a              | O
▁student        | O
▁in             | O
▁HCM            | B-ORG
US              | I-ORG
.               | O


# Evaluation

In [36]:
dataset = load_dataset('/kaggle/input/datasets/kimngntrn510/input-train/data/processed/conll2003_mapped')

## Tokenize and Align Input before predict

In [42]:
def tokenize_and_align_labels(examples):
    # 1. Để tokenizer tự thêm <s>, </s> (tương đương CLS, SEP) và padding
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        max_length=128,
        add_special_tokens=True
    )

    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        
        for word_idx in word_ids:
            # Token đặc biệt (<s>, </s>, <pad>) có word_id là None
            if word_idx is None:
                label_ids.append(-100)
            # Nếu là sub-token đầu tiên của một từ
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            # Nếu là các sub-token sau của cùng một từ
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs



In [43]:
# Tokenize and align dataset
test_dataset = dataset["test"].map(tokenize_and_align_labels, batched=True)

columns_to_keep = ["input_ids", "attention_mask", "labels"]
columns_to_remove = [col for col in test_dataset.column_names if col not in columns_to_keep]
test_dataset = test_dataset.remove_columns(columns_to_remove)

test_dataset.set_format(type="torch")

## Define metric for evaluation

In [49]:
seqeval_metric = evaluate.load("seqeval")

label_list = list(id2label.values())

data_collator = DataCollatorForTokenClassification(tokenizer)
test_dataloader = DataLoader(
    test_dataset, 
    batch_size=8, 
    collate_fn=data_collator
)

# 2. Định nghĩa hàm tính toán
def compute_metrics(p):
    predictions, labels = p
    
    # predictions là mảng xác suất (logits), ta lấy argmax để ra ID nhãn cao nhất
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    # Dùng seqeval để tính toán
    results = seqeval_metric.compute(predictions=true_predictions, references=true_labels)
    
    # Trả về dictionary các chỉ số
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

## Batch evaluate

In [54]:
from tqdm.auto import tqdm

def batch_evaluate(dataloader, model, device="cuda"):
    model.eval()
    
    all_logits = []
    all_labels = []
    global_max_len = 0
    
    print("Đang thu thập dữ liệu từ các Batch...")
    
    for batch in tqdm(dataloader, desc="Dự đoán (Inference)"):
        inputs = {k: v.to(device) for k, v in batch.items() if k != 'labels'}
        labels = batch['labels'].to(device)
        
        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits # Shape: [batch_size, seq_len, num_labels]
        
        # Chuyển về Numpy
        logits_np = logits.cpu().numpy()
        labels_np = labels.cpu().numpy()
        
        # Tìm chiều dài lớn nhất trong toàn bộ tập Test
        seq_len = logits_np.shape[1]
        if seq_len > global_max_len:
            global_max_len = seq_len
            
        all_logits.append(logits_np)
        all_labels.append(labels_np)
        
    print("Đang căn chỉnh độ dài (Padding) và tính toán Metrics...")
    
    # Pad các batch ngắn cho bằng global_max_len để có thể gộp thành 1 mảng duy nhất
    padded_logits = []
    padded_labels = []
    
    for logit, label in tqdm(zip(all_logits, all_labels), total=len(all_logits), desc="Căn chỉnh (Padding)"):
        seq_len = logit.shape[1]
        pad_length = global_max_len - seq_len
        
        # Pad cho Logits: Thêm 0 vào chiều seq_len (chiều thứ 1)
        pad_logit = np.pad(logit, ((0, 0), (0, pad_length), (0, 0)), constant_values=0)
        # Pad cho Labels: Thêm -100 vào chiều seq_len (chiều thứ 1)
        pad_label = np.pad(label, ((0, 0), (0, pad_length)), constant_values=-100)
        
        padded_logits.append(pad_logit)
        padded_labels.append(pad_label)
        
    # Gộp tất cả lại thành 1 mảng Numpy khổng lồ giống hệt format của Trainer
    final_logits = np.concatenate(padded_logits, axis=0)
    final_labels = np.concatenate(padded_labels, axis=0)
    
    # 4. GỌI HÀM COMPUTE_METRICS CỦA BẠN
    # Truyền vào Tuple (predictions, labels) đúng như hàm mong đợi
    metrics = compute_metrics((final_logits, final_labels))
    
    return metrics, final_logits, final_labels

In [55]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

results, predictions, references = batch_evaluate(test_dataloader, model, device)

print("\n" + "="*35)
print(f"KẾT QUẢ ĐÁNH GIÁ (CUSTOM LOOP)")
print("="*35)
print(f"F1 Tổng    : {results['f1']:.4f}")
print(f"Accuracy   : {results['accuracy']:.4f}")
print(f"Precision  : {results['precision']:.4f}")
print(f"Recall     : {results['recall']:.4f}")

Đang thu thập dữ liệu từ các Batch...


Dự đoán (Inference):   0%|          | 0/864 [00:00<?, ?it/s]

Đang căn chỉnh độ dài (Padding) và tính toán Metrics...


Căn chỉnh (Padding):   0%|          | 0/864 [00:00<?, ?it/s]


KẾT QUẢ ĐÁNH GIÁ (CUSTOM LOOP)
F1 Tổng    : 0.9114
Accuracy   : 0.9833
Precision  : 0.8990
Recall     : 0.9241
